Запуск SAM3 через ultralytics

In [12]:
from ultralytics.models.sam import SAM3SemanticPredictor
import cv2
import numpy as np

Сегментация контактных площадок

In [7]:
# Initialize predictor with configuration
overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="/mnt/ssd/users/alexb/hf/models/sam3/sam3.pt",
    half=True,  # Use FP16 for faster inference
    save=True,
)
predictor = SAM3SemanticPredictor(overrides=overrides)

# Set image once for multiple queries
predictor.set_image("example.jpg")

# Query with a single concept
results = predictor(text=["small silver element"])

Ultralytics 8.4.6 🚀 Python-3.10.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA RTX A5000, 24248MiB)
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
  Cloning https://github.com/ultralytics/CLIP.git to /tmp/pip-req-build-gbux7mgt
  Running command git clone --filter=blob:none --quiet https://github.com/ultralytics/CLIP.git /tmp/pip-req-build-gbux7mgt
  Resolved https://github.com/ultralytics/CLIP.git to commit 643beff3883b5720d94b6b9c9eca12fa9fb72fb1
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1382997 sha256=7a6d516e38e59163dd3718353f48611b98ff84f50ee65f24609983c9b05a74d3
  Store

In [15]:

def extract_and_save_masks(results, output_prefix='output'):
    """
    Извлекает маски из результатов Ultralytics SAM 3 и сохраняет два изображения:
    1. Слой только с сегментированными объектами (на прозрачном фоне)
    2. Оригинальное изображение без этих объектов (с прозрачными дырками)
    
    Args:
        results: результат model.predict() от Ultralytics
        output_prefix: префикс для имён выходных файлов
    """
    # Получаем данные из results
    masks_tensor = results[0].masks.data
    orig_img = results[0].orig_img  # BGR формат
    height, width = orig_img.shape[:2]
    
    # 1. Создаём объединённую маску всех объектов
    combined_mask = np.zeros((height, width), dtype=np.uint8)
    for mask_tensor in masks_tensor:
        mask_np = mask_tensor.cpu().numpy().astype(np.uint8) * 255
        combined_mask = cv2.bitwise_or(combined_mask, mask_np)
    
    # 2. Слой ТОЛЬКО с объектами (серебристый цвет)
    layer_objects = np.zeros((height, width, 4), dtype=np.uint8)
    object_color = [192, 192, 192, 255]  # Серебристый, непрозрачный
    
    for c in range(3):
        layer_objects[:, :, c] = np.where(combined_mask > 0, object_color[c], 0)
    layer_objects[:, :, 3] = combined_mask
    
    # 3. Оригинал БЕЗ объектов (с прозрачностью вместо них)
    orig_with_alpha = cv2.cvtColor(orig_img, cv2.COLOR_BGR2BGRA)
    inverted_mask = cv2.bitwise_not(combined_mask)
    orig_with_alpha[:, :, 3] = cv2.bitwise_and(
        orig_with_alpha[:, :, 3], 
        inverted_mask
    )
    
    # 4. Сохранение
    cv2.imwrite(f'{output_prefix}_objects_layer.png', layer_objects)
    cv2.imwrite(f'{output_prefix}_without_objects.png', orig_with_alpha)
    
    print(f"Сохранено:")
    print(f"1. {output_prefix}_objects_layer.png - слой с объектами")
    print(f"2. {output_prefix}_without_objects.png - оригинал без объектов")
    
    return layer_objects, orig_with_alpha, combined_mask

# Использование:
extract_and_save_masks(results, 'contact_pads')

Сохранено:
1. contact_pads_objects_layer.png - слой с объектами
2. contact_pads_without_objects.png - оригинал без объектов


(array([[[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]],
 
        [[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]],
 
        [[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]],
 
        ...,
 
        [[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]],
 
        [[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]],
 
        [[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]]], shape=(600, 600, 4), dtype=uint8),
 array([[[ 55, 149,  